In [6]:
pip install langchain langchain-groq langchain-community chromadb sentence-transformers langchain_text_splitters


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Load and Chunk the Document

In [7]:
# !pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the file you just created
loader = TextLoader("../platform-knowledge.txt")
documents = loader.load()

# Split text into chunks so the AI can search effectively
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from your text file.")


Created 11 chunks from your text file.


In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embedding model (runs on your CPU)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the Vector Database and save it to a folder named 'db'
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory="../chroma_db"
)

print("Vector Database created and saved to ../chroma_db")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8052.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Database created and saved to ../chroma_db


In [ ]:
query = "How can i view support contact?"
docs = vector_db.similarity_search(query, k=2)
# docs = [doc for doc, score in docs if score < 0.5]
context = "\n\n".join([doc.page_content for doc in docs])

from langchain_groq import ChatGroq
from app.core.config import GROQ_API_KEY
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.2   # ✅ more precise answers
)

prompt = f"""
You are an AI assistant for an Ed-Tech platform.

Use ONLY the context below to answer the question.

Rules:
- Keep answer short (max 2-3 lines)
- Be precise
- No extra information

Context:
{context}

Question: {query}

Answer:
"""

# ✅ Step 5: Generate answer
response = llm.invoke(prompt)

print(response.content)



For technical issues with video playback or OTP delivery, contact support@edtech-app.com. The platform is available 24/7 for global learners.
